# ¿Vale la pena un Vision Transformer para clasificar residuos?
## Comparación de redes convolucionales contra Vision Transformers, pensando en un sistema embebido

**Gabriela Medina Navidad**
Proyecto final de Inteligencia Artificial (DCI019), Doctorado en Ciencias e Ingeniería, Universidad Autónoma de Guadalajara.
Profesor: Dr. Eduardo Ulises Moya Sánchez.
Código: [github.com/gnavidad1978/clasificacion_residuos_ia](https://github.com/gnavidad1978/clasificacion_residuos_ia)

## 1. El problema

Mi tesis doctoral trata de un bote de basura inteligente para el campus. Cada bote tiene una cámara: la persona acerca su residuo, el sistema reconoce de qué material es y le indica si ese es el bote correcto **antes** de abrir la compuerta. Así se evita que se mezclen los residuos y, de paso, la gente aprende a separar.

Para que funcione en el campus, el modelo tiene que correr dentro del propio bote, en una computadora pequeña como una Raspberry Pi, alimentada por un panel solar. Eso significa que no basta con que el modelo acierte: también tiene que ser **ligero y rápido**.

Las categorías de mi tesis salen de la norma de Jalisco para separar residuos, la **NAE-SEMADES-007/2008** [9]: vidrio, plástico, metal, papel y cartón, residuos inorgánicos de difícil reciclaje y orgánico. En este proyecto uso cinco de ellas más una clase de "otros"; en la sección 4 explico por qué.

## 2. La pregunta que quise responder

Para clasificar imágenes hoy hay dos grandes familias de redes neuronales:

| Familia | Cómo "mira" una foto | Fama |
|---|---|---|
| **Red convolucional (CNN)** | Recorre la imagen con pequeñas lupas que detectan bordes, luego texturas, luego partes y al final objetos completos | La opción clásica, probada y eficiente |
| **Vision Transformer (ViT)** | Corta la foto en cuadritos, como un rompecabezas, y compara cada cuadrito con todos los demás para decidir qué es | Lo más moderno, pero pide más datos y más cómputo |

Siguiendo la recomendación del profesor, mi pregunta fue: **¿un Vision Transformer da un desempeño mejor que justifique su costo, o una CNN logra lo mismo con mucho menos cómputo?** Para un bote con panel solar, esa diferencia de costo importa mucho.

Como un transformer puede rendir distinto por ser transformer o simplemente por ser grande, comparé dos de ellos: uno grande (ViT-B/16) y uno pequeño (DeiT-Small), este último de un tamaño parecido al de ResNet-50.

No entrené ninguna red desde cero. Usé **aprendizaje por transferencia**: tomé redes que ya habían aprendido a reconocer formas y texturas con más de un millón de fotos de la base ImageNet [7], les cambié la última capa para que respondieran con mis seis categorías y las volví a entrenar con fotos de residuos. Es como contratar a alguien que ya sabe ver y solo enseñarle el trabajo nuevo.

| Modelo | Familia | Parámetros | Por qué lo elegí |
|---|---|---|---|
| MobileNetV3-Large [3] | CNN | 4.2 millones | Diseñada para teléfonos y equipos pequeños |
| ResNet-50 [2] | CNN | 23.5 millones | La referencia estándar, con conexiones residuales |
| YOLO11n-cls [5] | CNN | 1.5 millones | Es de la familia YOLO, la misma del detector que usaré en mi tesis |
| DeiT-Small [10] | Transformer | 21.7 millones | Un transformer del tamaño de ResNet-50, para separar el efecto de la familia del efecto del tamaño |
| ViT-B/16 [4] | Transformer | 85.8 millones | El Vision Transformer base |

Los dos transformers tienen pesos preentrenados solo con ImageNet-1K, igual que las CNN.

## 3. Cómo organicé el proyecto

Seguí la misma organización que vimos en clase: el código vive en un repositorio de GitHub con control de versiones, y este cuaderno solo ejecuta comandos. Cada experimento se configura en un archivo de texto YAML, así que para cambiar de modelo o de parámetros no toco el código.

| Archivo | Qué hace |
|---|---|
| `prepare_data.py` | Descarga las fotos, las acomoda en mis categorías y las separa en grupos |
| `model.py` | Contiene los cinco modelos, tomados de torchvision, Ultralytics y timm [11] (una "fábrica de modelos") |
| `dataset.py` | Lee las fotos y las prepara para la red |
| `main.py` | Entrena con PyTorch Lightning [8] leyendo el YAML |
| `configs/*.yaml` | Un archivo de configuración por modelo |
| `benchmark.py` | Mide cuánto cómputo necesita cada modelo |
| `report.py` | Arma la tabla comparativa y las gráficas |
| `evaluate_test.py` | Hace la evaluación final con el grupo de prueba, una sola vez |

Esta primera celda prepara todo: conecta mi Google Drive, donde guardo las fotos y los resultados; baja mi repositorio de GitHub; instala las librerías con sus versiones fijas, para que el cuaderno dé lo mismo aunque salgan versiones nuevas, y carga todas las librerías que uso. También define qué modelos comparo y con qué semillas. Entrené cada modelo **tres veces** con semillas distintas (42, 7 y 2026), porque una sola corrida puede salir bien o mal por suerte; con tres puedo reportar el promedio y cuánto varía. La línea que empieza con un código de siete caracteres es la versión exacta del código que se está usando.

In [ ]:
USUARIO_GITHUB = 'gnavidad1978'
RUTA_DRIVE = '/content/drive/MyDrive/proyecto_IA'
MODELOS = {
    'mobilenet_v3_large': 'configs/mobilenet_v3.yaml',
    'yolo11n_cls': 'configs/yolo11n_cls.yaml',
    'resnet50': 'configs/resnet50.yaml',
    'deit_small': 'configs/deit_small.yaml',
    'vit_b_16': 'configs/vit_b_16.yaml',
}
SEMILLAS = [42, 7, 2026]

# Drive, repositorio e instalacion de librerias con versiones fijas
from google.colab import drive
drive.mount('/content/drive')
%cd /content
!test -d clasificacion_residuos_ia && git -C clasificacion_residuos_ia pull -q || git clone -q https://github.com/{USUARIO_GITHUB}/clasificacion_residuos_ia.git
%cd /content/clasificacion_residuos_ia
!git log --oneline -1
!pip install -q -r requirements.txt

# Todas las importaciones
import os
import glob
import json
import pandas as pd
import torch, torchvision, lightning, timm, ultralytics
from IPython.display import Image, display
print('PyTorch', torch.__version__, '| torchvision', torchvision.__version__, '| Lightning', lightning.__version__,
      '| timm', timm.__version__, '| ultralytics', ultralytics.__version__,
      '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'sin GPU')

# Los resultados se guardan en Drive para no perderlos al cerrar Colab
os.makedirs(f'{RUTA_DRIVE}/repo_resultados', exist_ok=True)
os.makedirs(f'{RUTA_DRIVE}/repo_runs', exist_ok=True)
if not os.path.islink('resultados'):
    !rm -rf resultados
    !ln -s "{RUTA_DRIVE}/repo_resultados" resultados
if not os.path.exists('runs'):
    !cp -r "{RUTA_DRIVE}/repo_runs" runs

## 4. Los datos

Como mi propio conjunto de fotos del campus todavía está en construcción, para este proyecto usé **RealWaste**, un conjunto público de 4,752 fotografías de residuos reales, recolectados en un relleno sanitario de Australia [1]. Lo elegí porque sus residuos están sucios, aplastados y deformados, como la basura de verdad, y no son productos limpios sobre una cartulina blanca. Además trae residuos orgánicos, todas sus fotos tienen el mismo tamaño (524 por 524 píxeles) y su licencia permite usarlo citándolo. Cada residuo está fotografiado solo, al centro, sobre un piso de concreto.

RealWaste tiene nueve clases, que acomodé así:

| Clase en RealWaste | Mi categoría |
|---|---|
| Cardboard y Paper | Papel y cartón |
| Glass | Vidrio |
| Metal | Metal |
| Plastic | Plástico |
| Food Organics y Vegetation | Orgánico |
| Miscellaneous Trash | Otros (ver la revisión de abajo) |
| Textile Trash | La excluí: en el campus no habrá bote de textiles |

Me quedaron 4,434 fotos.

### La revisión de "Miscellaneous Trash"

Al principio asigné Miscellaneous Trash a "residuos inorgánicos de difícil reciclaje", pero era una suposición mía, así que la revisé: elegí 50 fotos al azar del grupo de entrenamiento y clasifiqué cada una. Solo una era claramente de difícil reciclaje según la norma (un tenedor desechable), y como mucho siete lo serían. El resto eran plásticos, metales, papel, aparatos electrónicos, textiles, pañuelos sucios y materiales que mi sistema no contempla, como cerámica, hule o cables. Es decir, en RealWaste esta clase es "todo lo que no cupo en otra".

Por eso no la presento como difícil reciclaje, sino como una clase de **otros**. La conservé para no cambiar la separación de las fotos ni repetir todos los entrenamientos, y lo declaro como limitación. Para mi tesis es un hallazgo importante: confirma que los conjuntos públicos no tienen una categoría equivalente a la de difícil reciclaje de la norma de Jalisco.

In [ ]:
if os.path.exists(f'{RUTA_DRIVE}/revision_misc_50.png'):
    display(Image(f'{RUTA_DRIVE}/revision_misc_50.png', width=700))

### Las reglas que seguí para que el experimento sea confiable

Antes de entrenar seguí cuatro reglas del protocolo experimental del curso.

**1. Semilla fija.** Todo lo que se hace "al azar" (cómo se reparten las fotos, en qué orden se muestran) depende de una semilla. Al fijarla, cualquiera que corra este cuaderno obtiene exactamente la misma separación de las fotos. El entrenamiento en GPU puede variar un poco aun con la misma semilla, porque la tarjeta gráfica hace algunas operaciones en un orden que no siempre es el mismo; por eso, además, repetí cada entrenamiento con tres semillas y reporto el promedio y la variación.

**2. Separar los datos antes de mirarlos.** Antes de ver cualquier foto, las separé en tres grupos: 70 % para entrenar, 15 % para validar y 15 % para la prueba final. Es como un examen: si el modelo estudia con las mismas preguntas que después le van a calificar, la calificación no dice nada. El grupo de **validación** lo uso para decidir cuándo parar y qué versión del modelo quedarme; el de **prueba** lo guardé sin tocar para el final.

Además cuidé un detalle: si dos fotos son casi idénticas, las mantuve siempre en el mismo grupo. Para detectarlas calculé una especie de "huella digital" de cada foto y comparé las huellas. Así el modelo no puede "reconocer" en la validación una foto que ya vio al entrenar.

**3. Comparar contra una línea base.** El modelo más tonto posible es uno que siempre contesta la categoría más común. Si mis modelos no le ganan por mucho, no aprendieron nada.

**4. Elegir la métrica antes de ver resultados.** Lo explico en la sección 5.

In [ ]:
!python prepare_data.py --zip "{RUTA_DRIVE}/realwaste.zip"
display(Image('resultados/eda/conteo_entrenamiento.png'))
display(Image('resultados/eda/muestras_entrenamiento.png', width=600))

### Lo que encontré al explorar los datos

Todas las fotos de entrenamiento son JPEG, a color y del mismo tamaño, así que no tuve que corregir formatos mezclados. Las categorías **no tienen la misma cantidad de fotos**: la más grande tiene más del doble que la más chica (vidrio es la que menos tiene). Eso influyó en cómo elegí la métrica.

La línea base, que siempre contesta "papel y cartón", obtuvo una **exactitud de 0.217** pero un **F1 macro de 0.059**.

## 5. Cómo medí el desempeño

Si solo midiera la **exactitud** (cuántas fotos acierta en total), un modelo podría verse bien aunque ignore las categorías con pocas fotos. Por eso elegí como métrica principal el **F1 macro**: calcula qué tan bien le va al modelo en cada categoría por separado y después las promedia, de modo que vidrio pesa lo mismo que papel. En el bote, equivocarse con vidrio es tan grave como equivocarse con plástico.

La línea base lo deja muy claro: con exactitud parece que acierta más que el azar (0.217 contra 1/6 = 0.167), pero su F1 macro de 0.059 muestra que en realidad no sabe nada. Cualquier modelo mío tiene que estar muy por encima de 0.059.

También reviso la **matriz de confusión**, una tabla que dice qué categoría se confundió con cuál.

## 6. El entrenamiento

Entrenar es mostrarle al modelo las fotos de entrenamiento una y otra vez. Cada vuelta completa se llama **época**. En cada época el modelo comete errores, mide qué tan equivocado está (la **pérdida**) y ajusta sus pesos para equivocarse menos. Al final de cada época lo califico con las fotos de validación, que nunca usa para aprender.

Configuré estas reglas en el YAML de cada modelo:

| Parámetro | Valor | Por qué |
|---|---|---|
| Épocas máximas | 15 | Un tope para limitar el costo de cada corrida |
| Paro temprano | Si el F1 de validación no mejora en 4 épocas | Para no seguir entrenando cuando ya no aprende y empieza a memorizar |
| Mejor modelo | Me quedo con la época de mejor F1 en validación, no con la última | Es la versión que mejor generaliza |
| Tamaño de imagen | 224 por 224 píxeles | Es el tamaño con el que se preentrenaron estas redes |
| Tasa de aprendizaje | 0.0003 en las CNN, 0.00003 en los transformers | Los transformers se desestabilizan con tasas altas |
| Aumento de datos | Volteo horizontal, giro de hasta 15° y cambios de brillo, contraste y saturación, solo en entrenamiento | Para que el modelo tolere distintas luces y posiciones |

No usé volteo vertical, porque en el bote nadie va a sostener su residuo de cabeza. Para que la comparación fuera justa, entrené todos los modelos con las mismas fotos, los mismos aumentos y las mismas reglas de paro; la única diferencia de receta es la tasa de aprendizaje de los transformers. No ajusté los parámetros de cada modelo por separado. Por ejemplo, el archivo `configs/vit_b_16.yaml` contiene, entre otras, estas líneas:

```yaml
trainer:
  max_epochs: 15
model:
  backbone: vit_b_16
  lr: 0.00003
data:
  img_size: 224
```

Para observar el entrenamiento usé **TensorBoard**, que dibuja las curvas de pérdida y de F1 de cada corrida.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

Esta celda entrena los cinco modelos con las tres semillas, es decir, quince entrenamientos. Cada uno es un solo comando, por ejemplo `python main.py fit --config configs/vit_b_16.yaml --seed_everything 7`. Si un entrenamiento ya está guardado en mi Drive, lo salta, así que puedo continuar donde me quedé.

In [ ]:
for backbone, config in MODELOS.items():
    for semilla in SEMILLAS:
        if os.path.exists(f'resultados/{backbone}_semilla{semilla}.json'):
            print(f'{backbone}, semilla {semilla}: ya entrenado')
            continue
        comando = f'python main.py fit --config {config} --seed_everything {semilla}'
        !{comando}
        respaldo = f'cp -r runs/. "{RUTA_DRIVE}/repo_runs/"'
        !{respaldo}

## 7. Cuánto cuesta cada modelo

Al principio comparé el **tiempo total de entrenamiento**, pero descubrí que no servía: las CNN y el ViT-B/16 tardaron casi lo mismo, unos 8 minutos, aunque el ViT tiene 57 veces más parámetros que YOLO. Incluso por época, en la GPU, las tres CNN tardaron entre 33 y 38 segundos y el ViT 53. Había dos razones. La primera, que en Colab gratuito solo hay dos procesadores para abrir y preparar las fotos, y ese paso tarda lo mismo para cualquier red; la GPU pasaba buena parte del tiempo esperando las fotos. La segunda, que cada modelo se detuvo en una época distinta.

Para medir el costo real de cada red, en esta celda le doy a cada modelo datos ya cargados en memoria y cuento **cuántas fotos por segundo puede procesar**, al entrenar y al clasificar. Así solo cuenta el cómputo de la arquitectura. Hice esta medición en el procesador de Colab, porque se me agotó la cuota de GPU. Los números sirven para comparar los modelos entre sí, no como tiempos reales del bote: el procesador de Colab es de otro tipo (x86, dos núcleos) que el de una Raspberry Pi 5 (ARM, cuatro núcleos). La celda solo mide los modelos que todavía no tienen resultado guardado.

Durante el entrenamiento también medí el tiempo de clasificar **una foto a la vez en el procesador**, con 4 hilos. Por la misma razón, es una referencia para comparar modelos; el tiempo en la Raspberry Pi lo mediré en el aparato real como parte de mi tesis.

In [ ]:
ruta_costo = 'resultados/costo_computo.csv'
medidos = pd.read_csv(ruta_costo, index_col='modelo').index.tolist() if os.path.exists(ruta_costo) else []
faltan = [m for m in MODELOS if m not in medidos]
if faltan:
    lista = ' '.join(faltan)
    !python benchmark.py --modelos {lista}
print(pd.read_csv(ruta_costo, index_col='modelo').round(1).to_string())

| Modelo | Fotos por segundo al entrenar | Fotos por segundo al clasificar |
|---|---|---|
| YOLO11n-cls | 22.4 | 81.1 |
| MobileNetV3-Large | 7.7 | 33.4 |
| DeiT-Small | 1.7 | 5.5 |
| ResNet-50 | 1.4 | 4.6 |
| ViT-B/16 | 0.5 | 1.6 |

Aquí sí aparece la diferencia que el tiempo total escondía. Con el cómputo aislado, el ViT-B/16 es unas 15 veces más lento que MobileNetV3 para entrenar y unas 21 veces más lento para clasificar; frente a YOLO la diferencia es de más de 40 veces. ResNet-50 clasifica unas 7 veces más lento que MobileNetV3. DeiT-Small y ResNet-50, que tienen casi el mismo número de parámetros, cuestan prácticamente lo mismo. El orden coincide con el número de parámetros de cada red.

## 8. Resultados

In [ ]:
!python report.py
for nombre in ['curvas', 'confusion_mobilenet_v3_large', 'confusion_resnet50', 'confusion_deit_small',
               'confusion_vit_b_16', 'confusion_yolo11n_cls', 'errores_mobilenet_v3_large', 'errores_vit_b_16']:
    ruta = f'resultados/figuras/{nombre}.png'
    if os.path.exists(ruta):
        display(Image(ruta, width=1000 if nombre == 'curvas' or nombre.startswith('errores') else 430))

### Lo que obtuve

Estos son los resultados en el grupo de validación, como promedio ± desviación estándar de las tres semillas:

| Modelo | Familia | F1 macro | Parámetros | Tamaño | Tiempo por foto en CPU |
|---|---|---|---|---|---|
| ResNet-50 | CNN | 0.962 ± 0.008 | 23.5 millones | 94 MB | 160 ms |
| MobileNetV3-Large | CNN | 0.957 ± 0.004 | 4.2 millones | 17 MB | 46 ms |
| ViT-B/16 | Transformer | 0.939 ± 0.008 | 85.8 millones | 343 MB | 532 ms |
| DeiT-Small | Transformer | 0.938 ± 0.007 | 21.7 millones | 87 MB | 155 ms |
| YOLO11n-cls | CNN | 0.913 ± 0.003 | 1.5 millones | 6 MB | 24 ms |
| Línea base | | 0.059 | | | |

Todos los modelos aprendieron: están muy por encima de la línea base de 0.059. Para decidir si dos modelos son realmente distintos uso una regla sencilla: divido la diferencia entre sus promedios entre la desviación más grande de los dos. Si el resultado es menor que 1, sus corridas se enciman y no puedo decir cuál es mejor; si es claramente mayor, la diferencia difícilmente es suerte. Con tres corridas por modelo, cualquier prueba estadística formal tendría poca fuerza, así que uso esta regla como guía y no como prueba.

| Comparación | Diferencia | Desviación mayor | Diferencia entre desviación |
|---|---|---|---|
| ResNet-50 contra MobileNetV3 | 0.005 | 0.008 | 0.6: empate |
| ViT-B/16 contra DeiT-Small | 0.001 | 0.008 | 0.1: empate |
| MobileNetV3 contra ViT-B/16 | 0.018 | 0.008 | 2.3 |
| MobileNetV3 contra DeiT-Small | 0.019 | 0.007 | 2.7 |
| ResNet-50 contra ViT-B/16 | 0.023 | 0.008 | 2.9 |
| ResNet-50 contra DeiT-Small | 0.024 | 0.008 | 3.0 |
| MobileNetV3 contra YOLO11n-cls | 0.044 | 0.004 | 11 |

**El ViT-B/16 no ganó; quedó por debajo.** Obtuvo unos dos puntos menos que ResNet-50 y MobileNetV3, entre 2 y 3 veces la desviación mayor. Y es, por mucho, el más costoso: 20 veces más parámetros que MobileNetV3 y más de 11 veces más lento para clasificar una foto.

**DeiT-Small respondió la duda del tamaño.** Tiene casi el mismo número de parámetros y el mismo costo que ResNet-50 (21.7 contra 23.5 millones; 155 contra 160 ms por foto), y aun así quedó 2.4 puntos abajo, empatado con el ViT-B/16, que es cuatro veces más grande. Es decir, con un tamaño comparable, la CNN siguió ganando: la diferencia no se debe solo a que el ViT-B/16 sea grande.

**ResNet-50 y MobileNetV3 empataron.** Con el mismo desempeño, MobileNetV3 es casi seis veces más pequeña y más de tres veces más rápida. Es el mejor balance entre costo y desempeño, y la candidata natural para la Raspberry Pi.

**YOLO11n-cls es el más ligero y el más rápido, pero también el más bajo**, unos cuatro a cinco puntos debajo. Su resultado fue muy estable entre semillas.

### Lo que muestran las curvas

En todos los modelos la pérdida de entrenamiento llega casi a cero mientras la de validación se estanca: el modelo empieza a memorizar las fotos de entrenamiento. El paro temprano lo detectó y me quedé con la mejor época. El ViT-B/16 fue el que sobreajustó más rápido: en promedio se detuvo a las 9 épocas, porque su mejor resultado llegó pronto. MobileNetV3, ResNet-50 y DeiT-Small entrenaron entre 13 y 14 épocas. YOLO, en cambio, llegó al tope de 15 épocas en las tres semillas, es decir, seguía mejorando de vez en cuando; es posible que con más épocas alcance un mejor resultado. YOLO es también el modelo con mayor separación entre su curva de entrenamiento y la de validación.

### Dónde se equivocan

Este es el F1 de cada categoría, promedio de las tres semillas:

| Categoría | MobileNetV3 | ResNet-50 | DeiT-Small | ViT-B/16 | YOLO11n-cls |
|---|---|---|---|---|---|
| Vidrio | 0.979 | 0.977 | 0.966 | 0.961 | 0.961 |
| Plástico | 0.945 | 0.949 | 0.920 | 0.936 | 0.893 |
| Metal | 0.941 | 0.939 | 0.944 | 0.924 | 0.884 |
| Papel y cartón | 0.971 | 0.980 | 0.944 | 0.958 | 0.930 |
| Otros (Miscellaneous Trash) | **0.922** | **0.938** | **0.876** | **0.877** | **0.844** |
| Orgánico | 0.984 | 0.990 | 0.981 | 0.980 | 0.966 |

Los cinco modelos coinciden: la categoría más difícil es **otros** (en ResNet-50 prácticamente empatada con metal). La segunda más difícil es **metal** en cuatro de los modelos y **plástico** en DeiT-Small. Si todos fallan en el mismo lugar, el problema no parece ser la arquitectura sino la categoría, y la revisión de la sección 4 lo explica: "otros" es una mezcla de materiales muy distintos, que se parecen a casi todas las demás categorías.

El otro error frecuente fue confundir **metal con plástico**. Al revisar fotos mal clasificadas en una corrida de prueba encontré aerosoles con su tapa de plástico a la vista, latas con etiquetas impresas y papel aluminio arrugado confundido con papel. Mi hipótesis es que el modelo se guía sobre todo por la apariencia de la superficie y no por el material de abajo, lo que coincidiría con lo que vimos en clase sobre el sesgo de las CNN hacia la textura [6]; no lo comprobé con un experimento, así que lo presento como explicación posible.

También encontré fotos donde la etiqueta original de RealWaste no coincide con la norma de Jalisco. La norma coloca el papel aluminio, las envolturas de golosinas y los envases multicapa tipo Tetra Pak en difícil reciclaje, mientras que RealWaste los etiqueta como metal, plástico o cartón.

## 9. Conclusiones

Respondiendo a mi pregunta: **para este problema, los Vision Transformers que probé no justificaron su costo.** El ViT-B/16 es el modelo más pesado y más lento, y obtuvo menor desempeño que las dos mejores redes convolucionales. DeiT-Small, un transformer del tamaño de ResNet-50, tampoco las alcanzó: con un costo casi igual al de ResNet-50, quedó 2.4 puntos abajo. Por eso la desventaja no se explica solo por el tamaño. Esta conclusión vale para estos dos transformers, preentrenados solo con ImageNet-1K y entrenados con la misma receta; no la generalizo a todos los transformers. MobileNetV3 alcanzó el mismo desempeño que ResNet-50 con una fracción del tamaño y del tiempo, por lo que es la mejor opción para un bote que funcionará con energía solar.

### Limitaciones

Es importante decir lo que estos resultados **no** demuestran.

| Limitación | Por qué importa |
|---|---|
| RealWaste no tiene residuos sostenidos en la mano | En el bote la gente acercará el residuo en la mano y con el campus de fondo; espero que el desempeño baje en el entorno real |
| La clase "otros" no es difícil reciclaje | Miscellaneous Trash es una mezcla; mis resultados cubren cinco categorías de la norma más una clase residual |
| Los transformers se preentrenaron solo con ImageNet-1K | Los transformers dependen más que las CNN de un preentrenamiento grande; con pesos preentrenados en más datos podrían rendir mejor |
| La misma receta para todos | No ajusté los parámetros de cada modelo; YOLO llegó al tope de épocas y quizá mejore con más épocas o con su propia receta |
| Tres semillas con la misma separación | Miden cuánto varía el entrenamiento, no cuánto cambiaría el resultado con otra separación de los datos |
| Fotos del mismo objeto desde otro ángulo | RealWaste no indica qué fotos son del mismo objeto; algunas pudieron quedar en grupos distintos y hacer que la validación se vea un poco mejor |
| El tiempo en la Raspberry Pi | Lo comparé en el procesador de Colab, que es de otro tipo; hay que medirlo en el aparato real |

### Qué sigue

La evaluación con el grupo de prueba, que mantuve guardado durante todo el proyecto, se hace una sola vez al final (sección 10). Después falta probar el mejor modelo con fotos tomadas en el campus de residuos sostenidos en la mano. En esas fotos el residuo ocupa una parte pequeña de la imagen, así que primero usaré un detector YOLO para recortarlo y después lo clasificaré con la CNN. Medir la degradación del modelo entre el entorno de laboratorio y el entorno real es justamente lo que busca mi tesis.

## 10. Evaluación final con el grupo de prueba

Esta evaluación la reservé para el final y la hago **una sola vez**, cuando ya no voy a cambiar modelos ni parámetros. Evalúa la mejor versión guardada de cada una de las quince corridas, así que también reporta promedio y desviación entre semillas. Si ya se hizo, el programa no la repite y solo muestra los resultados guardados.

In [ ]:
# Se ejecuta una sola vez, al final del proyecto. Quitar el signo # para correrla:
# !python evaluate_test.py

## Referencias

[1] S. Single, S. Iranmanesh y R. Raad, "RealWaste: A Novel Real-Life Data Set for Landfill Waste Classification Using Deep Learning," *Information*, vol. 14, no. 12, p. 633, 2023, doi: 10.3390/info14120633. Conjunto de datos en UCI Machine Learning Repository, doi: 10.24432/C5SS4G.

[2] K. He, X. Zhang, S. Ren y J. Sun, "Deep Residual Learning for Image Recognition," en *Proc. IEEE Conf. Computer Vision and Pattern Recognition (CVPR)*, 2016, pp. 770–778.

[3] A. Howard *et al.*, "Searching for MobileNetV3," en *Proc. IEEE/CVF Int. Conf. Computer Vision (ICCV)*, 2019, pp. 1314–1324.

[4] A. Dosovitskiy *et al.*, "An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale," en *Proc. Int. Conf. Learning Representations (ICLR)*, 2021.

[5] G. Jocher y J. Qiu, *Ultralytics YOLO11*, versión 11.0.0, 2024. [En línea]. Disponible: https://github.com/ultralytics/ultralytics

[6] R. Geirhos, P. Rubisch, C. Michaelis, M. Bethge, F. A. Wichmann y W. Brendel, "ImageNet-trained CNNs are biased towards texture; increasing shape bias improves accuracy and robustness," en *Proc. Int. Conf. Learning Representations (ICLR)*, 2019.

[7] J. Deng, W. Dong, R. Socher, L.-J. Li, K. Li y L. Fei-Fei, "ImageNet: A Large-Scale Hierarchical Image Database," en *Proc. IEEE Conf. Computer Vision and Pattern Recognition (CVPR)*, 2009, pp. 248–255.

[8] W. Falcon y The PyTorch Lightning team, *PyTorch Lightning*, 2019, doi: 10.5281/zenodo.3828935.

[9] Secretaría de Medio Ambiente para el Desarrollo Sustentable, "Norma Ambiental Estatal NAE-SEMADES-007/2008," *Periódico Oficial El Estado de Jalisco*, núm. 35, secc. III, 16 de octubre de 2008.

[10] H. Touvron, M. Cord, M. Douze, F. Massa, A. Sablayrolles y H. Jégou, "Training data-efficient image transformers & distillation through attention," en *Proc. Int. Conf. Machine Learning (ICML)*, PMLR vol. 139, 2021, pp. 10347–10357.

[11] R. Wightman, *PyTorch Image Models*, 2019. [En línea]. Disponible: https://github.com/huggingface/pytorch-image-models